# List Webull Sandbox Accounts

This notebook calls Webull's account-list endpoint. It does not require `WEBULL_ACCOUNT_ID` and does not place orders.

Install the optional SDK first if needed: `python -m pip install -e '.[webull]'`.

In [1]:
import json
import os
from pathlib import Path


def load_dotenv_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        name, value = line.split("=", 1)
        name = name.strip()
        value = value.strip().strip("'").strip('\"')
        os.environ.setdefault(name, value)


load_dotenv_file(Path.cwd() / ".env")

required = ("WEBULL_APP_KEY", "WEBULL_APP_SECRET")
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise RuntimeError(f"Missing environment variable(s): {', '.join(missing)}")

region = os.environ.get("WEBULL_REGION", "us")
endpoint = os.environ.get("WEBULL_API_ENDPOINT", "api.sandbox.webull.com")
print(f"Using Webull region={region!r}, endpoint={endpoint!r}")

Using Webull region='us', endpoint='api.sandbox.webull.com'


In [2]:
from webull.core.client import ApiClient
from webull.trade.trade_client import TradeClient

api_client = ApiClient(
    os.environ["WEBULL_APP_KEY"],
    os.environ["WEBULL_APP_SECRET"],
    region,
)
api_client.add_endpoint(region, endpoint)
trade_client = TradeClient(api_client)
response = trade_client.account_v2.get_account_list()

if response.status_code != 200:
    raise RuntimeError(
        f"Webull account-list request failed: HTTP {response.status_code}; "
        f"body={response.text}"
    )

payload = response.json() or {}
print(json.dumps(payload, indent=2, sort_keys=True))

135162148988736 2026-09-23 03:32:12,252 webull.core.http.initializer.client_initializer INFO _check_token_enable result is False
[
  {
    "account_class": "EVENTS_CASH",
    "account_id": "1EG00R4VT0A0883D1FM4JF1FC9",
    "account_label": "Events Cash",
    "account_number": "DEL3PQQ6",
    "account_type": "CASH",
    "user_id": "1180021118"
  },
  {
    "account_class": "INDIVIDUAL_CASH",
    "account_id": "85IOH3TSVMBGAPU770H931SNO8",
    "account_label": "Individual Cash",
    "account_number": "DEL7PRV6",
    "account_type": "CASH",
    "user_id": "1180021118"
  },
  {
    "account_class": "INDIVIDUAL_MARGIN",
    "account_id": "J0HCID6MGGRC0EI0N0363358B8",
    "account_label": "Individual Margin",
    "account_number": "DEL9PRQ3",
    "account_type": "MARGIN",
    "user_id": "1180021118"
  },
  {
    "account_class": "FUTURES",
    "account_id": "SIGA4295DPE45S6B965B78EM89",
    "account_label": "Futures",
    "account_number": "DEL9PRP2",
    "account_type": "MARGIN",
    "user_

In [3]:
def account_records(payload):
    value = payload.get("data", payload) if isinstance(payload, dict) else payload
    if isinstance(value, dict):
        value = value.get("accounts", value.get("items", value))
    if isinstance(value, dict):
        value = [value]
    return value if isinstance(value, list) else []


records = account_records(payload)
account_ids = []
for record in records:
    if not isinstance(record, dict):
        continue
    account_id = (
        record.get("account_id")
        or record.get("accountId")
        or record.get("id")
    )
    if account_id is not None:
        account_ids.append(str(account_id))

if account_ids:
    print("Available account IDs:")
    for account_id in account_ids:
        print(account_id)
else:
    print("No account IDs were found in the response. Inspect the raw payload above.")

Available account IDs:
1EG00R4VT0A0883D1FM4JF1FC9
85IOH3TSVMBGAPU770H931SNO8
J0HCID6MGGRC0EI0N0363358B8
SIGA4295DPE45S6B965B78EM89
X822952256743497728
